# **Preprocesamiento de audio**
## **Sistema de clasificación de sonidos urbanos para alertas**

**Proyecto:** Detección de sonidos de emergencia en entornos urbanos  
**Equipo:** Alessandra · Natalia · Andrés

### **¿Qué hace este notebook?**

Este notebook tiene dos objetivos:

1. **Ejecutar el preprocesado** de los ≥60.200 audios usando la clase `Preprocess` (`src/data/preprocess.py`) para generar los espectrogramas y features escalares que necesita el modelo.
2. **EDA post-preprocesado**: analizar y visualizar el dataset procesado (`processed_metadata.csv`) para verificar la calidad del proceso antes de entrenar.

### **Salidas que genera el preprocesador**

| Archivo | Descripción |
|---|---|
| `mel_path` | Mel-spectrogram en dB --> `.npy` `[1, 128, T]` |
| `mfcc_path` | Coeficientes MFCC --> `.npy` `[1, 40, T]` |
| `waveform_path` | Waveform normalizada --> `.npy` `[1, samples]` |
| `processed_metadata.csv` | CSV con rutas y features escalares |
| `label_mapping_human_label.pkl` | Mapeo clase --> índice entero |
| `label_mapping_alertable.pkl` | Mapeo alertable --> índice entero |

## **1. Instalación de dependencias**

In [ ]:
pip install librosa torchaudio soundfile tqdm matplotlib seaborn pandas numpy

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import librosa
import librosa.display
import torch
import torchaudio
import torchaudio.transforms as T
import pickle
import json
import os

from torch.utils.data import Dataset, DataLoader
from pathlib import Path
from multiprocessing import Pool, cpu_count
from functools import partial
from tqdm import tqdm
from sklearn.preprocessing import LabelEncoder
from typing import Dict, List, Tuple

from src.data.preprocess import Preprocess, PreprocessConfig
from src.data.build.metadata import MetadataEX
import src.data.build.zenodo_ds.zenodo_ds as zenodo
from src.data.build.dataset import Dataset
from src.utils.config import RAW_DIR, INTERIM_DIR

import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Estilo de gráficos
sns.set_theme(style='darkgrid', palette='muted')
plt.rcParams.update({
    'figure.dpi': 120,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
})

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Dispositivo: {DEVICE}')
if DEVICE == 'cuda':
    print(f'{torch.cuda.get_device_name(0)}')
print(f'PyTorch {torch.__version__} · torchaudio {torchaudio.__version__}')

## **2. Configuración de rutas**

In [ ]:
PROCESSED_DIR = INTERIM_DIR / 'processed_dataset'
METADATA_CSV = PROCESSED_DIR / 'processed_metadata.csv'
LABEL_MAP_HUMAN = PROCESSED_DIR / 'label_mapping_human_label.pkl'
LABEL_MAP_ALERT = PROCESSED_DIR / 'label_mapping_alertable.pkl'

print('Rutas del proyecto:')
print(f'RAW_DIR --> {RAW_DIR}')
print(f'INTERIM_DIR --> {INTERIM_DIR}')
print(f'PROCESSED --> {PROCESSED_DIR}')
print(f'METADATA CSV --> {METADATA_CSV}')

## **3. Ejecución del preprocesado**

- La clase `Preprocess` realiza las siguientes transformaciones por cada audio:

```
Audio raw  -->  Mono  -->  Resample a 44100 Hz  -->  Pad/Crop a 5s  -->  Normalización peak
                                                                           │
                                          ┌────────────────────────────────┤
                                          ▼                ▼               ▼
                                   mel_db.npy        mfcc.npy       waveform.npy
                                   [1, 128, T]       [1, 40, T]      [1, samples]
```

- **Procesado incremental:** Si los `.npy` ya existen para un audio, el proceso lo omite y pasa al siguiente.

In [ ]:
# CSVs de entrada
CSV_FILES = [
    'UrbanSound8k.csv',
    'audioset.csv',
    'ESC50.csv',
    'zenodo.csv',
    'Guns_DS.csv',
    'VOICe.csv',
    'ravdess_dataset.csv',
]
csv_paths = [RAW_DIR / f for f in CSV_FILES]

# Configuración
config = PreprocessConfig(
    sample_rate=44100,
    n_mels=128,
    n_mfcc=40, # 40 coeficientes para la rama MFCC del modelo dual
    n_fft=2048,
    hop_length=512,
    target_duration=5.0,
    normalize_peak=True,
    augment=False,
    save_waveform=True,
    save_mel=True,
    save_mfcc=True,
)

# Instanciar y ejecutar el preprocesado
pp = Preprocess(
    csv_paths=csv_paths,
    raw_dir=RAW_DIR,
    interim_dir=INTERIM_DIR,
    config=config,
)

print('Configuración activa:')
for k, v in vars(config).items():
    print(f'{k:30s}: {v}')


In [ ]:
# EJECUTAR PREPROCESADO
# Tiempo estimado: ~30 min en CPU.
# Los audios ya procesados se saltan automáticamente.

df_processed = pp.run()

print(f'\nPreprocesado completado.')
print(f'Registros totales en histórico: {len(df_processed):,}')
print(f'Columnas generadas: {list(df_processed.columns)}')
df_processed.head()

---
## **4. EDA post-preprocesado**
- Una vez generado el CSV, analizamos el dataset resultante para detectar posibles problemas antes de entrenar.

### **4.1 Carga del dataset procesado**

In [ ]:
df = pd.read_csv(METADATA_CSV)

print(f'Shape: {df.shape}')
print(f'Filas: {len(df):,}')
print(f'Columnas: {df.shape[1]}')

In [ ]:
print('Tipos de datos:')
print(df.dtypes.to_string())

In [ ]:
df.head()

In [ ]:
df.tail()

In [ ]:
# Valores nulos
nulls = df.isnull().sum()
nulls = nulls[nulls > 0].sort_values(ascending=False)

if nulls.empty:
    print('No hay valores nulos en el dataset procesado.')
else:
    print(f'Columnas con nulos ({len(nulls)}):')
    pct = (nulls / len(df) * 100).round(2)
    display(pd.DataFrame({'nulos': nulls, '%': pct}))